# Structural Coverage Exercise

In this exercise, we'll practice:
- **Instrumentation** for testing (tracking coverage)
- **Coverage calculation** (statement, branch, condition, MCDC, and path coverage)
- **Manual test suite generation** to achieve high coverage

This hands-on approach will help you deeply understand how structural coverages work.

## Software under Test (SuT)

We'll test a function `classify_triangle` that classifies triangles based on three side lengths `a`, `b`, and `c`.

The function returns:
- `"invalid"` - if the sides don't form a valid triangle
- `"equilateral"` - if all three sides are equal
- `"right_isosceles"` - if it's both a right triangle and isosceles
- `"isosceles"` - if exactly two sides are equal
- `"right"` - if it's a right triangle (satisfies Pythagorean theorem)
- `"scalene"` - if all sides are different

In [30]:
def classify_triangle(a: int, b: int, c: int) -> str:
    """Classify a triangle based on three side lengths."""
    sides = sorted([a, b, c])
    x, y, z = sides  # x <= y <= z

    if x <= 0:
        return "invalid"  # Negative or zero length is not allowed

    if x + y <= z:
        return "invalid"  # Violates triangle inequality

    # From here, we have a valid triangle
    if x == y == z:
        return "equilateral"

    is_isosceles = (x == y) or (y == z)
    is_right = (x * x + y * y == z * z)

    if is_isosceles and is_right:
        return "right_isosceles"
    elif is_isosceles:
        return "isosceles"
    elif is_right:
        return "right"
    else:
        return "scalene"

## Instrumented SuT

To calculate test coverage, the SuT should be **instrumented** or **observed** during execution.

Tools like `PyTest` support automatic coverage tracking by:
1. Parsing the source code (AST manipulation)
2. Inserting tracking statements at strategic points
3. Recording which code elements were executed
4. Generating coverage reports
However, the tools typically support only some of coverages. 
(e.g., PyTest only supports statement branch coverage.)

In this exercise, we'll implement manual instrumentation to:
- understand how coverage tracking works internally
- Track all types of coverage: statement, branch, condition, MCDC and path
- See exactly what data needs to be collected for each coverage type

### Implementation Approach

We'll implement a `CoverageTracker` base class that will be the parent class for each type of coverage tracker.

#### `CoverageTracker` Responsibilities:

- **Contains** the instrumented SuT (instrumented `classify_triangle`)
- **Resets** coverage data between test runs
- **Runs** the SuT and tracks coverage for each test input
- **Calculates** coverage metrics
- **Reports** coverage results

Each child class (`StatementCoverageTracker`, `BranchCoverageTracker`, etc.) will inherit from `CoverageTracker` and implement its specific tracking logic.


In [ ]:
from abc import ABC, abstractmethod
from typing import List, Tuple
from copy import deepcopy
from tabulate import tabulate


class CoverageTracker(ABC):
    """
    Abstract base class for coverage tracking.
    
    Stores coverage as: [(input, coverage_list, result), ...]
    where coverage_list is [(item_id, bool), ...]
    """
    
    def __init__(self):
        """Initialize the coverage tracker."""
        self.zero_coverage = []  # Template: [(item_id, False), ...]
        self.executions = []  # List of (input, coverage, result) triples
        self._initialize_zero_coverage()
        self.reset()
    
    @abstractmethod
    def _initialize_zero_coverage(self):
        """
        Initialize the zero coverage template.
        Must be implemented by child classes.
        
        Example for statements:
            self.zero_coverage = [("stm1", False), ("stm2", False), ...]
        """
        pass
    
    def reset(self):
        """Reset execution history."""
        self.executions = []
    
    def run_test(self, a: int, b: int, c: int) -> str:
        """
        Run the instrumented SuT and record coverage.
        
        Args:
            a, b, c: Triangle side lengths
            
        Returns:
            str: Classification result
        """
        test_input = (a, b, c)
        coverage, result = self.instrumented_classify_triangle(a, b, c)
        self.executions.append((test_input, coverage, result))
        return result
    
    @abstractmethod
    def instrumented_classify_triangle(self, a: int, b: int, c: int) -> Tuple[List[Tuple[str, bool]], str]:
        """
        Instrumented version of classify_triangle.
        
        Args:
            a, b, c: Triangle side lengths
            
        Returns:
            Tuple[List[Tuple[str, bool]], str]: (coverage_list, result)
            where coverage_list is [(item_id, covered), ...]
        """
        pass
    
    def calculate_coverage(self) -> Tuple[float, int, int]:
        """
        Calculate overall coverage from all executions.
        
        Returns:
            Tuple[float, int, int]: (coverage_percentage, covered_items, total_items)
        """
        if not self.executions:
            return 0.0, 0, len(self.zero_coverage)
        
        # Aggregate coverage across all executions
        covered_items = set()
        for test_input, coverage, result in self.executions:
            for item_id, was_covered in coverage:
                if was_covered:
                    covered_items.add(item_id)
        
        total_items = len(self.zero_coverage)
        covered_count = len(covered_items)
        percentage = (covered_count / total_items * 100) if total_items > 0 else 0.0
        
        return percentage, covered_count, total_items
    
    def print_report(self):
        """Generate and print coverage report in table format using tabulate."""
        if not self.executions:
            print("No test executions recorded.")
            return
        
        # Calculate overall coverage
        percentage, covered, total = self.calculate_coverage()
        
        # Get coverage type name from class name
        coverage_type = self.__class__.__name__.replace("CoverageTracker", "").replace("Tracker", "")
        
        # Print header
        print("=" * 100)
        print(f"{coverage_type.upper()} COVERAGE REPORT")
        print("=" * 100)
        print(f"\nOverall Coverage: {percentage:.2f}% ({covered}/{total} items)")
        print(f"Total Tests: {len(self.executions)}\n")
        
        # Build table data
        headers = ["Test input"] + [test_input for test_input, _, _ in self.executions]
        
        # Coverage rows - use boolean directly
        table_data = []
        for item_id, _ in self.zero_coverage:
            row = [item_id] + [dict(coverage).get(item_id, False) for _, coverage, _ in self.executions]
            table_data.append(row)
        
        # Result row
        result_row = ["Result"] + [result for _, _, result in self.executions]
        
        # Print tables with double-line separator
        print(tabulate(table_data, headers=headers, tablefmt="grid"))
        print(tabulate([result_row], headers=headers, tablefmt="grid"))
        print("=" * 100)
    
    def get_coverage_percentage(self) -> float:
        """Get coverage percentage."""
        percentage, _, _ = self.calculate_coverage()
        return percentage
    
    def get_executions(self) -> List[Tuple[Tuple[int, int, int], List[Tuple[str, bool]], str]]:
        """
        Get all execution records.
        
        Returns:
            List of (input, coverage, result) triples
        """
        return self.executions

ModuleNotFoundError: No module named 'tabulate'

### Statement Coverage Tracker

**Statement Coverage** measures which executable statements have been executed at least once.

**Formula**: Statement Coverage = (Executed Statements / Total Statements) × 100%
The `StatementCoverageTracker` will:
- Track each statement (line) as it executes
- Store executed statement IDs in a set
- Calculate the percentage of statements covered



In [14]:
class StatementCoverageTracker(CoverageTracker):
    """
    Tracks statement coverage - which lines of code have been executed.
    """
    
    def _initialize_zero_coverage(self):
        """Initialize zero coverage for statements (excluding if/else lines)."""
        self.zero_coverage = [
            ("sides = sorted([a,b,c])", False),
            ("x, y, z = sides", False),
            ("return 'invalid' #1", False),      # x <= 0 check
            ("return 'invalid' #2", False),      # triangle inequality
            ("return 'equilateral'", False),
            ("is_isosceles = ...", False),
            ("is_right = ...", False),
            ("return 'right_isosceles'", False),
            ("return 'isosceles'", False),
            ("return 'right'", False),
            ("return 'scalene'", False),
        ]
    
    def instrumented_classify_triangle(self, a: int, b: int, c: int) -> Tuple[List[Tuple[str, bool]], str]:
        """
        Instrumented classify_triangle with statement tracking.
        Only tracks meaningful statements (not if/else lines).
        
        Args:
            a, b, c: Triangle side lengths
            
        Returns:
            Tuple[List[Tuple[str, bool]], str]: (coverage, result)
        """
        # Copy zero coverage to track this execution
        coverage_dict = dict(deepcopy(self.zero_coverage))
        
        # Statement: sort sides
        coverage_dict["sides = sorted([a,b,c])"] = True
        sides = sorted([a, b, c])
        
        # Statement: unpack
        coverage_dict["x, y, z = sides"] = True
        x, y, z = sides
        
        # Branch (not tracked as statement)
        if x <= 0:
            # Statement: return invalid #1
            coverage_dict["return 'invalid' #1"] = True
            return list(coverage_dict.items()), "invalid"
        
        # Branch (not tracked as statement)
        if x + y <= z:
            # Statement: return invalid #2
            coverage_dict["return 'invalid' #2"] = True
            return list(coverage_dict.items()), "invalid"
        
        # Branch (not tracked as statement)
        if x == y == z:
            # Statement: return equilateral
            coverage_dict["return 'equilateral'"] = True
            return list(coverage_dict.items()), "equilateral"
        
        # Statement: compute is_isosceles
        coverage_dict["is_isosceles = ..."] = True
        is_isosceles = (x == y) or (y == z)
        
        # Statement: compute is_right
        coverage_dict["is_right = ..."] = True
        is_right = (x * x + y * y == z * z)
        
        # Branch (not tracked as statement)
        if is_isosceles and is_right:
            # Statement: return right_isosceles
            coverage_dict["return 'right_isosceles'"] = True
            return list(coverage_dict.items()), "right_isosceles"
        elif is_isosceles:
            # Statement: return isosceles
            coverage_dict["return 'isosceles'"] = True
            return list(coverage_dict.items()), "isosceles"
        elif is_right:
            # Statement: return right
            coverage_dict["return 'right'"] = True
            return list(coverage_dict.items()), "right"
        else:
            # Statement: return scalene
            coverage_dict["return 'scalene'"] = True
            return list(coverage_dict.items()), "scalene"


In [20]:
# Example: Using StatementCoverageTracker

# Create tracker instance
stmt_tracker = StatementCoverageTracker()

# Run multiple tests
stmt_tracker.run_test(3, 4, 5)     # Right triangle
stmt_tracker.run_test(5, 5, 5)     # Equilateral
stmt_tracker.run_test(0, 1, 1)     # Invalid

# Print coverage report in table format
print()
stmt_tracker.print_report()


Running tests...

STATEMENT COVERAGE REPORT

Overall Coverage: 63.64% (7/11 items)
Total Tests: 3

Item                               | Test 1     | Test 2     | Test 3     |
                                   | (3, 4, 5)  | (5, 5, 5)  | (0, 1, 1)  |
-----------------------------------------------------------------------------
sides = sorted([a,b,c])            |      T     |      T     |      T     |
x, y, z = sides                    |      T     |      T     |      T     |
return 'invalid' #1                |      F     |      F     |      T     |
return 'invalid' #2                |      F     |      F     |      F     |
return 'equilateral'               |      F     |      T     |      F     |
is_isosceles = ...                 |      T     |      F     |      F     |
is_right = ...                     |      T     |      F     |      F     |
return 'right_isosceles'           |      F     |      F     |      F     |
return 'isosceles'                 |      F     |      F     | 